# LDV Cable Coupling — Modal Analysis

Slim driver: all reusable logic lives in the [`ldv_analysis/`](ldv_analysis) package
(`modal.py` for numerics, `modal_plotting.py` for figures). Edit the package, then
re-run the relevant cells here.

Each dataset is treated as one **cable + gap-length configuration** spanning a
doubly-clamped (clamped–clamped) gap. The pipeline finds transverse resonances,
extracts complex mode shapes \(\varphi_y, \varphi_z\), compares them to the
analytical Euler–Bernoulli beam modes via the **MAC**, and characterises the
**polarization** of each resonance.

| § | Content |
|---|---|
| 0 | Imports & plot style |
| 1 | Setup & configuration (datasets + detection parameters) |
| 2 | Spectral fingerprint & resonance detection (runs the full pipeline) |
| 3 | Mode-shape extraction \((\varphi_y, \varphi_z)\) |
| 4 | Theoretical comparison via MAC |
| 5 | Polarization analysis |
| 6 | Per-dataset deep-dive visualisations |
| 7 | Cross-dataset summary visualisations |
| 8 | Interactive exploration widget |

All per-dataset results are stashed in `cfg['modal']`, so every downstream cell
reads cached results — nothing is recomputed.


---
## § 0  Imports & Plot Style

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Editable reload while iterating on the package
%load_ext autoreload
%autoreload 2

import ldv_analysis as la
from ldv_analysis import config, io, modal, modal_plotting

plt.rcParams.update({
    "font.size": 12, "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "figure.dpi": 130, "savefig.dpi": 300, "figure.autolayout": True,
    "axes.titlesize": 11, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in",
})
print(f"{len(config.ALL_DATASETS)} datasets in catalogue.")


---
## § 1  Setup & Configuration

Pick the candidate datasets (`ACTIVE_LABELS`) and the modal-analysis parameters.

**Detection parameters** feed `scipy.signal.find_peaks` on the combined
total-power fingerprint:

| Parameter | Meaning |
|---|---|
| `F_MIN`, `F_MAX` | frequency search range [Hz] (default 5–500) |
| `PEAK_PROMINENCE` | peak prominence threshold, as a **fraction (0–1)** of the strongest in-band peak |
| `MIN_SPACING_HZ` | minimum spacing between detected resonances [Hz] |
| `POL_BW_FRAC` | bandpass fractional half-width for mode extraction / polarization (default 0.10 ⇒ \(f_n\pm10\%\)) |
| `N_THEORY_MODES` | number of clamped–clamped beam modes to compare against (MAC) |

> **Tuning note.** `PEAK_PROMINENCE` / `MIN_SPACING_HZ` are dataset-dependent.
> If a dataset over-detects (a dense comb of closely-spaced peaks), raise the
> prominence or the spacing — the **fingerprint plot in § 2 is the diagnostic**.


In [ ]:
# ── Candidate datasets — edit to taste (see config.ALL_DATASETS for all labels) ──
# One cable + gap-length configuration per entry.
ACTIVE_LABELS = [f"Cable{i}_10cm" for i in range(1, 8)]
# Examples:
# ACTIVE_LABELS = [f"Cable{i}_5cm"  for i in range(1, 8)]
# ACTIVE_LABELS = [f"Cable{i}_15cm" for i in range(1, 8)]
# ACTIVE_LABELS = ["Cable5_5cm", "Cable5_10cm", "Cable5_15cm"]   # one cable, gap sweep

# ── Detection / extraction parameters ───────────────────────────────────────
F_MIN, F_MAX     = 5.0, 500.0   # frequency search range [Hz]
PEAK_PROMINENCE  = 0.08         # fraction of the strongest in-band peak (0-1)
MIN_SPACING_HZ   = 10.0         # minimum spacing between resonances [Hz]
POL_BW_FRAC      = 0.10         # bandpass half-width for mode extraction (f_n +/- 10%)
N_THEORY_MODES   = 5            # clamped-clamped beam modes for MAC

# Polarization classification thresholds (axis ratio = minor/major)
LIN_THRESH, CIRC_THRESH = 0.10, 0.80   # <lin = linear, >circ = circular, else elliptical

DATASETS = config.select_datasets(ACTIVE_LABELS)
for cfg in DATASETS:
    io.load_cable_dataset(cfg, verbose=False)
    print(f"  {cfg['label']:<18} {cfg['n_sensors']:>2} sensors | gap {cfg['gap_m']*100:.0f} cm "
          f"| gap span sensors [{cfg['idx_left']}, {cfg['idx_right']}]")
print(f"\n{len(DATASETS)} dataset(s) loaded.")


---
## § 2  Spectral Fingerprint & Resonance Detection

`modal.analyze_dataset` runs the **entire pipeline** for each dataset and stores
the result in `cfg['modal']`:

1. **Fingerprint** — per-sensor amplitude spectra of \(v_y, v_z\) (over the gap
   sensors) collapsed into a per-polarization *max-amplitude* fingerprint and a
   *total-power* fingerprint \(\sum_s |V|^2\).
2. **Detection** — `find_peaks` on the combined total-power signal.
3. **Mode shapes** (§ 3), **MAC** (§ 4) and **polarization** (§ 5) — only at the
   detected peaks (efficient: nothing is computed at every frequency bin).

The fingerprint plot marks each detected resonance; use it to tune the § 1
detection parameters.


In [ ]:
for cfg in DATASETS:
    modal.analyze_dataset(
        cfg, f_min=F_MIN, f_max=F_MAX,
        prominence=PEAK_PROMINENCE, min_spacing_hz=MIN_SPACING_HZ,
        bw_frac=POL_BW_FRAC, n_theory_modes=N_THEORY_MODES,
        lin_thresh=LIN_THRESH, circ_thresh=CIRC_THRESH)


In [ ]:
# Spectral fingerprint per dataset (linear y; resonances marked)
for cfg in DATASETS:
    modal_plotting.plot_fingerprint(cfg)
    plt.show()


---
## § 3  Mode-Shape Extraction

For each detected resonance \(f_n\): bandpass \(v_y, v_z\) around
\(f_n(1\pm\)`POL_BW_FRAC`\()\), integrate to displacement (FFT integration),
then read the complex amplitude at the FFT bin nearest \(f_n\) — giving the
complex mode-shape vectors \(\varphi_y\) and \(\varphi_z\) over the gap sensors.

These are already computed and stored in `cfg['modal']['modes'][k]['phi_y' / 'phi_z']`.
The cell below is a sanity check.


In [ ]:
cfg = DATASETS[0]
md0 = cfg['modal']
print(f"{cfg['label']}: {len(md0['modes'])} resonance(s), "
      f"{len(md0['gap_idx'])} gap sensors (x/L in [{md0['xi'].min():.2f}, {md0['xi'].max():.2f}])\n")
for k, m in enumerate(md0['modes']):
    print(f"  [{k}] f_n = {m['f_n']:6.1f} Hz | "
          f"|phi_y| max = {np.abs(m['phi_y']).max()*1e6:7.2f} um | "
          f"|phi_z| max = {np.abs(m['phi_z']).max()*1e6:7.2f} um | "
          f"dominant component: {m['dom_component']}")


---
## § 4  Theoretical Comparison via MAC

The clamped–clamped Euler–Bernoulli mode shapes are

$$\varphi_n(\xi) = \big[\cosh\alpha_n\xi - \cos\alpha_n\xi\big]
   - \sigma_n\big[\sinh\alpha_n\xi - \sin\alpha_n\xi\big],\quad \xi = x/L,$$

with \(\sigma_n = (\cosh\alpha_n-\cos\alpha_n)/(\sinh\alpha_n-\sin\alpha_n)\) and
roots \(\alpha_n = [4.730, 7.853, 10.996, 14.137, 17.279]\).

Each measured mode (dominant polarization, complex) is compared to all
`N_THEORY_MODES` theoretical shapes via the **MAC**
\(= |a^{\mathsf H}b|^2/[(a^{\mathsf H}a)(b^{\mathsf H}b)]\); the best match gives
the assigned mode order. A global phase/scale cancels in the MAC, so node
sign-flips are handled correctly.


In [ ]:
# Theoretical mode-shape reference
xi = np.linspace(0, 1, 200)
th = modal.theoretical_modes(xi, N_THEORY_MODES)
fig, ax = plt.subplots(figsize=(8, 3.2))
for n in range(N_THEORY_MODES):
    ax.plot(xi, th[n], label=f'mode {n+1}')
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_xlabel('x / L'); ax.set_ylabel('norm. shape')
ax.set_title('Clamped-clamped Euler-Bernoulli mode shapes')
ax.legend(fontsize=8, ncol=5, loc='lower center')
plt.show()

print('f_n / f_1 ratios (theory):', np.round(modal.theoretical_fn_ratios(N_THEORY_MODES), 3))


In [ ]:
# MAC heatmap (measured resonances x theoretical modes) per dataset
for cfg in DATASETS:
    if cfg['modal']['modes']:
        modal_plotting.plot_mac_heatmap(cfg)
        plt.show()


---
## § 5  Polarization Analysis

For each resonance and sensor, the complex pair \((\varphi_y, \varphi_z)\) defines
a polarization ellipse, decomposed into co/counter-rotating circular components:

- **major / minor** axis amplitudes and **axis ratio** = minor/major,
- **orientation** \(\theta\) in the \(y\)–\(z\) plane,
- **handedness** = \(\operatorname{sign}\,\Im(\varphi_y\varphi_z^*)\).

Classification: *linear* (ratio < `LIN_THRESH`), *circular* (> `CIRC_THRESH`),
else *elliptical*. A single dominant label per resonance is the majority vote of
the amplitude-significant sensors.


In [ ]:
# Per-resonance dominant polarization summary
for cfg in DATASETS:
    print(f"{cfg['label']}:")
    for m in cfg['modal']['modes']:
        print(f"   {m['f_n']:6.1f} Hz | mode {m['mode_order']} (MAC {m['mac']:.2f}) | "
              f"{m['dom_pol']:<10} | axis ratio {m['dom_axis_ratio']:.3f} | "
              f"hand {m['dom_handedness']:+d}")


---
## § 6  Per-Dataset Deep-Dive Visualisations

For a chosen dataset:

- **`deep_dive`** — fingerprint + MAC heatmap + a 6-panel figure per resonance
  ((a) \(|\varphi_y|\), (b) \(|\varphi_z|\), (c) \(\angle\varphi_y\),
  (d) \(\angle\varphi_z\), (e) best theoretical overlay, (f) MAC/polarization summary)
  + the polarization ellipse map.
- **`plot_phase_snapshots`** — 3-D cable shape at four phases of one oscillation
  cycle for a *selected* resonance (call it explicitly only when you want it).


In [ ]:
# Pick a dataset to deep-dive (index into DATASETS or a label)
DEEP_DIVE_LABEL = DATASETS[0]['label']
cfg = next(c for c in DATASETS if c['label'] == DEEP_DIVE_LABEL)

modal_plotting.deep_dive(cfg)


In [ ]:
# Phase-snapshot montage for ONE selected resonance (3-D, 0/90/180/270 deg).
# Choose the resonance index within `cfg` (e.g. the lowest mode, or one flagged
# circular in the § 5 summary). Run only when you want this view.
RESONANCE_INDEX = 0
if cfg['modal']['modes']:
    modal_plotting.plot_phase_snapshots(cfg, RESONANCE_INDEX)
    plt.show()
else:
    print('No resonances detected for', cfg['label'])


---
## § 7  Cross-Dataset Summary Visualisations

- **Mode-frequency chart** — resonance frequency (log) per dataset, colour =
  mode order, marker = polarization (circle linear, square elliptical, star circular).
- **\(f_n\) vs gap length** (log–log) with theoretical \(L^{-2}\) reference lines
  per mode order; each point annotated with its cable.
- **Summary table** — one row per detected mode, exported to CSV.


In [ ]:
modal_plotting.plot_mode_frequency_chart(DATASETS)
plt.show()


In [ ]:
modal_plotting.plot_fn_vs_gap(DATASETS)
plt.show()


In [ ]:
summary = modal_plotting.build_summary_table(DATASETS)
summary.to_csv('modal_summary.csv', index=False)
print(f"{len(summary)} detected modes -> modal_summary.csv")
summary


---
## § 8  Interactive Exploration

Pick a dataset and a detected resonance, then switch between the fingerprint,
6-panel mode view, MAC heatmap, ellipse map, and 3-D phase snapshots.


In [ ]:
modal_plotting.modal_dashboard(DATASETS)
